# EigenAlpha Phase 0 — Full Pipeline Notebook

This notebook runs the complete EigenAlpha pipeline end-to-end, from data loading
through to tearsheet generation. It mirrors `pipeline.py` but in an interactive
format for step-by-step inspection.

## Pipeline Steps
1. Load & preprocess data
2. Covariance decomposition, PCA, clustering
3. Compute factors & IC analysis
4. Quintile backtests
5. Markowitz optimised backtest
6. Tearsheet generation

---
**Author:** EigenAlpha Research  
**Date:** Phase 0 (2024)

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(level=logging.INFO, format='%(name)-20s | %(message)s')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (14, 6)

print('Environment ready.')

## Step 1: Load Data

In [ ]:
from config import START_DATE, END_DATE, UNIVERSE_TICKERS, N_CLUSTERS
from data.loader import DataLoader
from data.preprocessor import Preprocessor
from data.universe import NIFTY500_TICKERS, SECTOR_MAP

loader = DataLoader()

try:
    raw_data = loader.load('../data_cache/nifty500_ohlcv.parquet')
    print(f'Loaded cached data: {raw_data.shape}')
except FileNotFoundError:
    print('Downloading data...')
    raw_data = loader.fetch(UNIVERSE_TICKERS, START_DATE, END_DATE)
    loader.save(raw_data, '../data_cache/nifty500_ohlcv.parquet')

# Extract prices and volumes
if isinstance(raw_data.columns, pd.MultiIndex):
    prices = raw_data['Close'] if 'Close' in raw_data.columns.get_level_values(0) else raw_data
    volumes = raw_data['Volume'] if 'Volume' in raw_data.columns.get_level_values(0) else pd.DataFrame()
else:
    prices = raw_data
    volumes = pd.DataFrame()

# Filter stocks with sufficient history
from config import MIN_HISTORY_MONTHS
min_obs = MIN_HISTORY_MONTHS * 21
valid = prices.columns[prices.notna().sum() >= min_obs]
prices = prices[valid]
if not volumes.empty:
    volumes = volumes[volumes.columns.intersection(valid)]

print(f'Prices: {prices.shape}')
print(f'Date range: {prices.index.min()} to {prices.index.max()}')

## Step 2: Preprocess

In [ ]:
pp = Preprocessor()
log_returns = pp.compute_log_returns(prices)
monthly_returns = pp.compute_monthly_returns(prices)
return_matrix = pp.build_return_matrix(log_returns)

print(f'Log returns: {log_returns.shape}')
print(f'Monthly returns: {monthly_returns.shape}')
print(f'Return matrix (R): {return_matrix.shape} — T={return_matrix.shape[0]}, N={return_matrix.shape[1]}')

## Step 3: PCA & Clustering

In [ ]:
from pca.decompose import CovarianceDecomposer
from pca.cluster import MarketClusterer

decomposer = CovarianceDecomposer(return_matrix)
decomposer.compute_covariance()
eigenvalues, eigenvectors = decomposer.eigendecompose()

n_comp = min(50, return_matrix.shape[1] - 1)
decomposer.fit_pca(n_components=n_comp)
k = decomposer.select_components(0.80)
print(f'\n{k} PCs explain 80% of variance')

# Scree plot
decomposer.plot_scree()
plt.show()

In [ ]:
clusterer = MarketClusterer(decomposer, n_clusters=N_CLUSTERS)
pc_scores = clusterer.get_stock_pc_scores(n_dims=3)
cluster_labels = clusterer.fit_kmeans()

print('\nCluster sizes:')
print(cluster_labels.value_counts().sort_index())

# Cluster plot
clusterer.plot_clusters_2d()
plt.show()

## Step 4: Compute Factors & IC Analysis

In [ ]:
from factors.engine import FactorEngine
from factors.ic_analysis import InformationCoefficient

engine = FactorEngine(prices, volumes)
factor_data = engine.compute_all()
print(f'Factor data: {factor_data.shape}')
factor_data.head()

In [ ]:
# Forward returns for IC
fwd_returns = monthly_returns.shift(-1)
fwd_long = fwd_returns.stack().reset_index()
fwd_long.columns = ['date', 'ticker', 'forward_return']

factor_with_fwd = factor_data.merge(fwd_long, on=['date', 'ticker'], how='inner')

# IC analysis
ic_results = {}
for factor_col in ['momentum_12_1', 'realized_vol', 'volume_trend']:
    ic = InformationCoefficient(
        factor_with_fwd,
        factor_with_fwd[['date', 'ticker', 'forward_return']]
    )
    summary = ic.ic_summary(factor_col)
    ic_results[factor_col] = summary
    
    print(f'{factor_col}: Mean IC = {summary["mean_ic"]:.4f}, IR = {summary["ir"]:.4f}, t = {summary["ic_t_stat"]:.2f}')
    
    # Plot
    ic.plot_ic_timeseries(factor_col)
    plt.show()

## Step 5: Quintile Backtests

In [ ]:
from portfolio.backtest import Backtester

benchmark = loader.get_benchmark('^NSEI', START_DATE, END_DATE)
bt = Backtester(prices, factor_data, benchmark)

for factor_col in ['momentum_12_1', 'realized_vol', 'volume_trend']:
    q_ret = bt.run_quintile_backtest(factor_col)
    
    # Plot quintile cumulative returns
    fig, ax = plt.subplots(figsize=(14, 5))
    for col in q_ret.columns:
        if col != 'Long_Short':
            cum = (1 + q_ret[col].fillna(0)).cumprod()
            ax.plot(cum.index, cum.values, label=col)
    ax.set_title(f'Quintile Returns — {factor_col}', fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.show()
    
    # Metrics
    if 'Long_Short' in q_ret.columns:
        m = bt.compute_metrics(q_ret['Long_Short'].dropna())
        print(f'{factor_col} L/S: Sharpe={m["sharpe_ratio"]:.2f}, Ann Ret={m["annualised_return"]*100:.1f}%')
    print()

## Step 6: Markowitz Optimised Backtest

In [ ]:
from portfolio.optimizer import MarkowitzOptimizer

markowitz_returns = bt.run_markowitz_backtest(
    optimizer_class=MarkowitzOptimizer,
    factor_col='momentum_12_1',
    lookback_months=36,
    cluster_labels=cluster_labels,
)

if len(markowitz_returns) > 0:
    metrics = bt.compute_metrics(markowitz_returns)
    print('\n=== Markowitz Portfolio ===')
    for k, v in metrics.items():
        print(f'  {k}: {v}')
    
    # Plot cumulative returns
    fig, ax = plt.subplots(figsize=(14, 5))
    cum = (1 + markowitz_returns).cumprod()
    ax.plot(cum.index, cum.values, linewidth=2, label='Markowitz Portfolio')
    
    # Add benchmark
    bench_monthly = benchmark.resample('ME').last().pct_change().dropna()
    common = cum.index.intersection(bench_monthly.index)
    if len(common) > 0:
        cum_bench = (1 + bench_monthly.loc[common]).cumprod()
        ax.plot(cum_bench.index, cum_bench.values, linewidth=1.5, linestyle='--', label='Nifty 50', alpha=0.7)
    
    ax.set_title('Markowitz vs Benchmark', fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.show()

## Step 7: Generate Tearsheet

In [ ]:
from portfolio.tearsheet import TearsheetGenerator

# Prepare IC results with ic_series for tearsheet
ic_full = {}
for factor_col in ['momentum_12_1', 'realized_vol', 'volume_trend']:
    ic_obj = InformationCoefficient(
        factor_with_fwd,
        factor_with_fwd[['date', 'ticker', 'forward_return']]
    )
    ic_series = ic_obj.compute_ic(factor_col)
    summary = ic_obj.ic_summary(factor_col)
    ic_full[factor_col] = {'ic_series': ic_series, **summary}

# Get quintile results for momentum
q_mom = bt.run_quintile_backtest('momentum_12_1')

portfolio_ret = markowitz_returns if len(markowitz_returns) > 0 else pd.Series([0.0], index=[pd.Timestamp('2020-01-31')])
bench_monthly = benchmark.resample('ME').last().pct_change().dropna()

ts = TearsheetGenerator(
    portfolio_returns=portfolio_ret,
    benchmark_returns=bench_monthly,
    factor_data=factor_data,
    ic_results=ic_full,
    quintile_returns=q_mom,
    pca_decomposer=decomposer,
    clusterer=clusterer,
    metrics=metrics if len(markowitz_returns) > 0 else {},
)

ts.generate('../outputs/tearsheet.png')
print('Tearsheet saved to outputs/tearsheet.png')

In [ ]:
print('\n' + '='*60)
print('  EigenAlpha Phase 0 — Pipeline Complete!')
print('='*60)